# 🚀 ARES: Kaggle Dual T4 7B LoRA Training & Multi-Domain Evaluation
**Adaptive Reliability with Expert Specialization (`Qwen2.5-7B-Instruct` 4-bit NF4)**

This notebook performs the **complete, end-to-end 7B LoRA activation and benchmark validation**:
1. **Dual T4 Hardware & Library Audit**: Validates 2x NVIDIA T4 GPUs (32 GB total VRAM), CUDA, and bitsandbytes 4-bit NF4 quantization.
2. **Repository Sync & Checkpoint Linking**: Clones the latest `ARES-research` codebase with router bypass fixes and links pre-trained GRM, LRM, and Router checkpoints from `/kaggle/input/`.
3. **PEFT LoRA Expert Fine-Tuning**: Trains genuine, non-zero HuggingFace PEFT LoRA adapters for 5 domains (**Math**, **Code**, **Science**, **Reasoning**, **General**) using 4-bit QLoRA on `Qwen2.5-7B-Instruct` (~8–10 minutes total).
4. **Interactive Diagnostic Divergence Check**: Evaluates GSM8K questions side-by-side on Base vs. Math LoRA Expert to verify non-zero divergence and CoT reasoning enhancement.
5. **Dual T4 Parallel Evaluation (500 Queries)**: Benchmarks all 5 routing strategies (`BASE`, `FIXED_EXPERT`, `DYNAMIC_ARES`, `THRESHOLD_ROUTER`, `RANDOM_ROUTER`) with smart caching.
6. **Empirical Table I & Statistical Tests**: Emits the final empirical LaTeX rows for Table I and computes paired Student's t-test and McNemar significance.


In [ ]:
# === [1/6] Kaggle Dual T4 Hardware & Environment Setup ===
import os
import sys
import torch

print('=' * 70)
print('  ARES KAGGLE DUAL T4 HARDWARE & ENVIRONMENT AUDIT')
print('=' * 70)

# 1. Verify Dual CUDA Devices
cuda_available = torch.cuda.is_available()
n_gpus = torch.cuda.device_count()
print(f'CUDA Available: {cuda_available}')
print(f'Detected GPUs:  {n_gpus}')

if cuda_available:
    total_vram = 0.0
    for i in range(n_gpus):
        gpu_name = torch.cuda.get_device_name(i)
        vram_gb = torch.cuda.get_device_properties(i).total_memory / (1024**3)
        total_vram += vram_gb
        print(f'  ✓ GPU {i}: {gpu_name} ({vram_gb:.2f} GB VRAM)')
    print(f'Total Available GPU VRAM: {total_vram:.2f} GB')
else:
    print('WARNING: CUDA is not active! In Kaggle sidebar: Settings -> Accelerator -> select "GPU T4 x2".')

# 2. Install / Verify Core Dependencies
!pip install -q --upgrade pip
!pip install -q 'transformers>=4.41.0' 'peft>=0.12.0' 'accelerate>=0.30.0' 'bitsandbytes>=0.43.0' datasets scipy tabulate

os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['TRANSFORMERS_NO_ADVISORY_WARNINGS'] = '1'
os.environ['PYTHONWARNINGS'] = 'ignore'
print('\n✓ Kaggle environment dependencies verified and ready!')


In [ ]:
# === [2/6] Repository Setup & Kaggle Input Linking ===
import os
import sys
import shutil
import zipfile
from pathlib import Path

# 1. Setup repository workspace in /kaggle/working
if Path('/kaggle/working').exists():
    os.chdir('/kaggle/working')
    if Path('ARES-research').exists():
        os.chdir('ARES-research')
        !git fetch origin
        !git reset --hard origin/main
        !git pull origin main
    else:
        !git clone https://github.com/sharksurfauto-byte/ARES-research.git
        os.chdir('ARES-research')
    repo_root = Path('/kaggle/working/ARES-research').resolve()
elif Path('src/ares').exists():
    repo_root = Path('.').resolve()
    !git pull origin main
else:
    !git clone https://github.com/sharksurfauto-byte/ARES-research.git
    os.chdir('ARES-research')
    repo_root = Path('.').resolve()

print(f'Active Workspace: {repo_root}')
if str(repo_root / 'src') not in sys.path:
    sys.path.insert(0, str(repo_root / 'src'))

# 2. Locate and link Pretrained Reliability Models & Router from /kaggle/input
ckpt_dir = repo_root / 'checkpoints'
ckpt_dir.mkdir(parents=True, exist_ok=True)

if not (ckpt_dir / 'reliability' / 'grm.pt').exists():
    print('Searching /kaggle/input for pre-trained GRM/LRM/Router checkpoints...')
    candidates = list(Path('/kaggle/input').rglob('grm.pt'))
    if candidates:
        source_dir = candidates[0].parent.parent
        print(f'Found checkpoints at {source_dir}! Linking reliability & router...')
        for subdir in ['reliability', 'router']:
            src_sub = source_dir / subdir
            dst_sub = ckpt_dir / subdir
            if src_sub.exists():
                shutil.copytree(src_sub, dst_sub, dirs_exist_ok=True)
    else:
        # Fallback to local zip archives
        for z in [Path('checkpoints.zip'), Path('/kaggle/working/checkpoints.zip'), Path('outputs/checkpoints.zip')]:
            if z.exists():
                print(f'Extracting {z} into {ckpt_dir}...')
                with zipfile.ZipFile(z, 'r') as zf:
                    zf.extractall(ckpt_dir)
                break

assert (ckpt_dir / 'reliability' / 'grm.pt').exists(), (
    'ERROR: checkpoints/reliability/grm.pt not found!\n'
    'Please click "+ Add Input" in the Kaggle sidebar and add dataset "ares-eval-input" '
    'or the previous notebook output containing the reliability checkpoints.'
)
assert (ckpt_dir / 'reliability' / 'lrm.pt').exists(), 'ERROR: checkpoints/reliability/lrm.pt missing!'
assert (ckpt_dir / 'router' / 'router.pt').exists() or (ckpt_dir / 'router' / 'router_best.pt').exists(), 'ERROR: router.pt missing!'

print('\n✓ PRETRAINED RELIABILITY & ROUTER VERIFIED:')
for p in sorted(ckpt_dir.rglob('*.pt')):
    if 'experts' not in str(p):
        print(f'   ✓ {p.relative_to(repo_root)} ({p.stat().st_size / (1024*1024):.2f} MB)')


In [ ]:
# === [3/6] Fine-Tune Genuine 7B PEFT LoRA Experts ===
# Trains genuine, non-zero domain adapters for math, code, science, reasoning, and general.
# Uses Qwen2.5-7B-Instruct in 4-bit NF4 with rank=16 (~1.5 min per domain, ~8 min total).

import os
import sys
import torch
from pathlib import Path

repo_root = Path('/kaggle/working/ARES-research').resolve() if Path('/kaggle/working/ARES-research').exists() else Path('.').resolve()
os.chdir(repo_root)

print('=' * 75)
print('  TRAINING GENUINE 7B PEFT LoRA EXPERTS (QLoRA 4-bit NF4)')
print('=' * 75)

# Run expert fine-tuning using --use_backbone to train real HuggingFace PEFT adapters
!python scripts/train_experts.py \
    --use_backbone \
    --model_name "Qwen/Qwen2.5-7B-Instruct" \
    --output_dir "checkpoints/experts" \
    --domains math code science reasoning general \
    --max_samples 100 \
    --batch_size 2 \
    --epochs 1 \
    --lr 2e-4 \
    --lora_r 16 \
    --lora_alpha 32 \
    --lora_dropout 0.05

print('\n' + '=' * 75)
print('  EXPERT ADAPTER VERIFICATION & WEIGHT AUDIT')
print('=' * 75)

domains = ['math', 'code', 'science', 'reasoning', 'general']
all_ok = True

for d in domains:
    exp_dir = repo_root / 'checkpoints' / 'experts' / d
    peft_weights = exp_dir / 'adapter_model.safetensors'
    peft_bin = exp_dir / 'adapter_model.bin'
    peft_cfg = exp_dir / 'adapter_config.json'
    pt_file = exp_dir / f'expert_{d}.pt'
    
    has_peft = (peft_weights.exists() or peft_bin.exists()) and peft_cfg.exists()
    
    # Calculate weight norm from .pt checkpoint
    pt_norm = 0.0
    if pt_file.exists():
        data = torch.load(pt_file, map_location='cpu', weights_only=False)
        sd = data.get('state_dict', {}) if isinstance(data, dict) else {}
        for p in sd.values():
            if hasattr(p, 'norm'):
                pt_norm += p.norm().item()
                
    status = '✓ VALID (Active Non-Zero Weights)' if (has_peft or pt_norm > 0.1) else '✗ FAIL (Zero Norm)'
    print(f'  Domain: {d:<10} | PEFT Saved: {has_peft!s:<5} | Weight Norm: {pt_norm:>8.4f} | {status}')
    if not (has_peft or pt_norm > 0.1):
        all_ok = False

if all_ok:
    print('\n✓ ALL 5 DOMAIN EXPERTS VERIFIED WITH ACTIVE NON-ZERO WEIGHTS!')
else:
    raise RuntimeError('One or more expert adapters failed to produce non-zero weights!')


In [ ]:
# === [4/6] Interactive Diagnostic Spot Check (Base vs Math Expert) ===
# Runs 5 GSM8K samples side-by-side to visually confirm divergence between Base and Math Expert.

import os
import sys
import torch
from pathlib import Path

repo_root = Path('/kaggle/working/ARES-research').resolve() if Path('/kaggle/working/ARES-research').exists() else Path('.').resolve()
if str(repo_root / 'src') not in sys.path:
    sys.path.insert(0, str(repo_root / 'src'))

from ares.pipeline import ARESPipeline, PipelineConfig
from ares.data.benchmark_loader import load_all_benchmark_samples
from ares.eval.extractors import extract_math_answer

print('=' * 75)
print('  DIAGNOSTIC SPOT CHECK: BASE MODEL vs MATH EXPERT')
print('=' * 75)

# Initialize pipeline on GPU 0
config = PipelineConfig(
    model_name="Qwen/Qwen2.5-7B-Instruct",
    load_in_4bit=True,
    device_map="cuda:0" if torch.cuda.is_available() else "cpu",
    fixed_expert_name="math",
    max_new_tokens=256,
)

pipeline = ARESPipeline(config=config)

# Load 5 GSM8K test samples
gsm8k_samples = load_all_benchmark_samples(n_samples_per_domain=5, split="test").get("math", [])

divergences = 0
for idx, sample in enumerate(gsm8k_samples):
    print(f'\n--- [Sample {idx+1}/{len(gsm8k_samples)}] ---')
    print(f'Question: {sample.prompt[:120]}...')
    print(f'Ground Truth Target: {sample.target_answer}')
    
    # Base generation
    res_base = pipeline.generate(prompt=sample.prompt, strategy="base", max_new_tokens=256)
    pred_base = extract_math_answer(res_base.generated_text)
    
    # Math Expert generation
    res_expert = pipeline.generate(prompt=sample.prompt, strategy="fixed_math", max_new_tokens=256)
    pred_expert = extract_math_answer(res_expert.generated_text)
    
    is_diff = (res_base.generated_text.strip() != res_expert.generated_text.strip())
    if is_diff:
        divergences += 1
        
    print(f'  [Base Completion]:   {res_base.generated_text[:100]}... -> Extracted: {pred_base}')
    print(f'  [Math Expert]:       {res_expert.generated_text[:100]}... -> Extracted: {pred_expert}')
    print(f'  -> Outputs Diverge:  {"YES (Divergent generation)" if is_diff else "NO (Identical)"}')

print('\n' + '=' * 75)
print(f'Divergence Rate: {divergences}/{len(gsm8k_samples)} ({divergences/len(gsm8k_samples)*100:.1f}%)')
if divergences > 0:
    print('✓ SUCCESS: Math Expert adapter is actively transforming generation outputs!')
else:
    print('WARNING: Zero divergence detected between Base and Math Expert. Check adapter hook.')


In [ ]:
# === [5/6] Full Multi-Domain Benchmark Evaluation across Dual T4 GPUs ===
# Evaluates 500 samples (100 per domain) across all baseline strategies.
# Parallelized across GPU 0 and GPU 1.

import os
import sys
import gc
import torch
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

repo_root = Path('/kaggle/working/ARES-research').resolve() if Path('/kaggle/working/ARES-research').exists() else Path('.').resolve()
if str(repo_root / 'src') not in sys.path:
    sys.path.insert(0, str(repo_root / 'src'))

from ares.pipeline import ARESPipeline, PipelineConfig
from ares.pipeline.baselines import BaselineComparator
from ares.data.benchmark_loader import load_all_benchmark_samples
from ares.eval.metrics import MetricsCalculator

# Configuration
SAMPLES_PER_DOMAIN = 100   # 100 samples x 5 domains = 500 queries total
MAX_NEW_TOKENS = 256
SPLIT = "test"

print('=' * 75)
print(f'  ARES 7B FULL BENCHMARK EVALUATION ({SAMPLES_PER_DOMAIN * 5} QUERIES)')
print('=' * 75)

# 1. Load multi-domain test samples
domain_dict = load_all_benchmark_samples(n_samples_per_domain=SAMPLES_PER_DOMAIN, split=SPLIT)
all_eval_samples = []
for d in ["math", "code", "science", "reasoning", "general"]:
    all_eval_samples.extend(domain_dict.get(d, []))

print(f'Total Evaluated Samples: {len(all_eval_samples)} across 5 domains.')

# 2. Parallel execution setup across Dual T4 GPUs
n_gpus = torch.cuda.device_count()
print(f'CUDA GPUs Available: {n_gpus}')

strategies = ['BASE', 'FIXED_EXPERT', 'DYNAMIC_ARES', 'THRESHOLD_ROUTER', 'RANDOM_ROUTER']

if n_gpus >= 2:
    print('\nInitializing Dual T4 GPUs: Pipeline 0 on cuda:0, Pipeline 1 on cuda:1...')
    cfg_0 = PipelineConfig(
        model_name="Qwen/Qwen2.5-7B-Instruct",
        load_in_4bit=True,
        device_map="cuda:0",
        fixed_expert_name="math",
        reliability_threshold=0.5,
        max_new_tokens=MAX_NEW_TOKENS,
    )
    cfg_1 = PipelineConfig(
        model_name="Qwen/Qwen2.5-7B-Instruct",
        load_in_4bit=True,
        device_map="cuda:1",
        fixed_expert_name="math",
        reliability_threshold=0.5,
        max_new_tokens=MAX_NEW_TOKENS,
    )
    
    pipe_0 = ARESPipeline(config=cfg_0)
    pipe_1 = ARESPipeline(config=cfg_1)
    
    comp_0 = BaselineComparator(pipe_0, strategies=strategies, fixed_expert="math", threshold=0.5)
    comp_1 = BaselineComparator(pipe_1, strategies=strategies, fixed_expert="math", threshold=0.5)
    
    # Interleave samples evenly
    batch_0 = [s for i, s in enumerate(all_eval_samples) if i % 2 == 0]
    batch_1 = [s for i, s in enumerate(all_eval_samples) if i % 2 == 1]
    print(f'  ✓ GPU 0 Batch: {len(batch_0)} samples | GPU 1 Batch: {len(batch_1)} samples')
    
    def cb_0(res, cur, tot):
        if cur % 10 == 0 or cur == tot:
            print(f'  [GPU 0] Evaluated {cur}/{tot} ({cur/tot*100:.1f}%)...', flush=True)
            
    def cb_1(res, cur, tot):
        if cur % 10 == 0 or cur == tot:
            print(f'  [GPU 1] Evaluated {cur}/{tot} ({cur/tot*100:.1f}%)...', flush=True)
            
    print('\nStarting parallel execution across Dual T4 GPUs (100% utilization on both)...')
    with ThreadPoolExecutor(max_workers=2) as executor:
        f_0 = executor.submit(comp_0.evaluate_batch, batch_0, max_new_tokens=MAX_NEW_TOKENS, verbose=False, checkpoint_callback=cb_0)
        f_1 = executor.submit(comp_1.evaluate_batch, batch_1, max_new_tokens=MAX_NEW_TOKENS, verbose=False, checkpoint_callback=cb_1)
        r_0 = f_0.result()
        r_1 = f_1.result()
        
    eval_results = []
    max_l = max(len(r_0), len(r_1))
    for i in range(max_l):
        if i < len(r_0): eval_results.append(r_0[i])
        if i < len(r_1): eval_results.append(r_1[i])

else:
    print('\nSingle GPU Mode...')
    cfg = PipelineConfig(
        model_name="Qwen/Qwen2.5-7B-Instruct",
        load_in_4bit=True,
        device_map="cuda:0" if torch.cuda.is_available() else "cpu",
        fixed_expert_name="math",
        reliability_threshold=0.5,
        max_new_tokens=MAX_NEW_TOKENS,
    )
    pipe = ARESPipeline(config=cfg)
    comp = BaselineComparator(pipe, strategies=strategies, fixed_expert="math", threshold=0.5)
    def cb(res, cur, tot):
        if cur % 10 == 0 or cur == tot:
            print(f'  Evaluated {cur}/{tot} ({cur/tot*100:.1f}%)...', flush=True)
    eval_results = comp.evaluate_batch(all_eval_samples, max_new_tokens=MAX_NEW_TOKENS, checkpoint_callback=cb)

# 3. Calculate and Save Report
metadata = {
    'model_name': "Qwen/Qwen2.5-7B-Instruct",
    'samples_per_domain': SAMPLES_PER_DOMAIN,
    'split': SPLIT,
    'max_new_tokens': MAX_NEW_TOKENS,
    'num_gpus_used': n_gpus,
}

report = MetricsCalculator.calculate_metrics(eval_results, metadata=metadata)
report.print_summary()

out_file = Path('outputs/benchmark_results_7b.json')
out_file.parent.mkdir(parents=True, exist_ok=True)
report.save_json(str(out_file))
print(f'\n✓ Saved 7B evaluation results to {out_file}')


In [ ]:
# === [6/6] Empirical Table I Formatter & Statistical Significance ===
# Formats the empirical run into LaTeX rows for Table I and computes statistical tests.

import os
import json
import numpy as np
from scipy import stats

print('=' * 75)
print('  EMPIRICAL TABLE I (LATEX ROWS FOR 7B BACKBONE)')
print('=' * 75)

domains = ["math", "code", "science", "reasoning", "general"]
strategies = ["BASE", "THRESHOLD_ROUTER", "RANDOM_ROUTER", "FIXED_EXPERT", "DYNAMIC_ARES"]

display_names = {
    "BASE": "Base Qwen2.5-7B (Zero-Shot)",
    "THRESHOLD_ROUTER": "Entropy / Threshold (7B)",
    "RANDOM_ROUTER": "Random Router (7B)",
    "FIXED_EXPERT": "Fixed Expert (Math 7B)",
    "DYNAMIC_ARES": r"\textbf{ARES 7B (Learned Router)}",
}

latex_rows = []

if "eval_results" in globals() and len(eval_results) > 0:
    for strat in strategies:
        strat_results = [r for r in eval_results if strat in r.results]
        n_total = len(strat_results)

        dom_accs = {}
        for d in domains:
            d_samples = [r for r in strat_results if r.domain == d]
            if d_samples:
                acc = sum(1 for r in d_samples if r.correctness.get(strat, False)) / len(d_samples) * 100.0
                dom_accs[d] = acc
            else:
                dom_accs[d] = 0.0

        overall_acc = sum(1 for r in strat_results if r.correctness.get(strat, False)) / n_total * 100.0 if n_total > 0 else 0.0
        inv_rate = sum(1 for r in strat_results if r.expert_invocations.get(strat, False)) / n_total * 100.0 if n_total > 0 else 0.0
        savings = 100.0 - inv_rate
        mean_lat = np.mean([r.latencies_ms.get(strat, 0.0) for r in strat_results]) if n_total > 0 else 0.0

        pct = r"\%"
        row_str = (
            f"{display_names.get(strat, strat):<35} & "
            f"{dom_accs['math']:>5.1f}{pct} & "
            f"{dom_accs['code']:>5.1f}{pct} & "
            f"{dom_accs['science']:>5.1f}{pct} & "
            f"{dom_accs['reasoning']:>5.1f}{pct} & "
            f"{dom_accs['general']:>5.1f}{pct} & "
            f"{overall_acc:>6.2f}{pct} & "
            f"{inv_rate:>5.1f}{pct} & "
            f"{savings:>5.1f}{pct} & "
            f"{mean_lat:>7.1f} ms \\\\"
        )
        latex_rows.append(row_str)

    print("\n".join(latex_rows))
    print('=' * 75)

    # Statistical Significance Testing: Base vs Dynamic ARES
    base_correct = [int(r.correctness.get("BASE", False)) for r in eval_results]
    ares_correct = [int(r.correctness.get("DYNAMIC_ARES", False)) for r in eval_results]

    if len(base_correct) > 0 and len(ares_correct) > 0:
        t_stat, p_val = stats.ttest_rel(ares_correct, base_correct)
        print()
        print("Statistical Significance (ARES vs Base):")
        print(f"  ✓ Paired Student t-statistic: t = {t_stat:.2f}, p-value = {p_val:.4e}")

        # McNemar test
        contingency = np.zeros((2, 2))
        for b, a in zip(base_correct, ares_correct):
            contingency[b, a] += 1
        b_wrong_a_right = contingency[0, 1]
        b_right_a_wrong = contingency[1, 0]
        if b_wrong_a_right + b_right_a_wrong > 0:
            chi2 = ((abs(b_wrong_a_right - b_right_a_wrong) - 1)**2) / (b_wrong_a_right + b_right_a_wrong)
        else:
            chi2 = 0.0
        print(f"  ✓ McNemar chi-square:        chi2 = {chi2:.2f} (discordants: {int(b_wrong_a_right)} ARES-only vs {int(b_right_a_wrong)} Base-only)")

elif os.path.exists("outputs/benchmark_results_7b.json"):
    with open("outputs/benchmark_results_7b.json", "r") as f:
        data = json.load(f)
    strat_metrics = data.get("strategy_metrics", {})
    for strat in strategies:
        m = strat_metrics.get(strat, {})
        dom_accs = m.get("domain_accuracies", {})
        overall_acc = m.get("accuracy", 0.0) * 100.0
        inv_rate = m.get("expert_invocation_rate", 0.0) * 100.0
        savings = 100.0 - inv_rate
        mean_lat = m.get("mean_latency_ms", 0.0)

        pct = r"\%"
        row_str = (
            f"{display_names.get(strat, strat):<35} & "
            f"{dom_accs.get('math', 0.0)*100.0:>5.1f}{pct} & "
            f"{dom_accs.get('code', 0.0)*100.0:>5.1f}{pct} & "
            f"{dom_accs.get('science', 0.0)*100.0:>5.1f}{pct} & "
            f"{dom_accs.get('reasoning', 0.0)*100.0:>5.1f}{pct} & "
            f"{dom_accs.get('general', 0.0)*100.0:>5.1f}{pct} & "
            f"{overall_acc:>6.2f}{pct} & "
            f"{inv_rate:>5.1f}{pct} & "
            f"{savings:>5.1f}{pct} & "
            f"{mean_lat:>7.1f} ms \\\\"
        )
        latex_rows.append(row_str)

    print("\n".join(latex_rows))
    print('=' * 75)
